# Classificação de Risco de Readmissão em 30 Dias

O objetivo é sinalizar, no momento da alta, quais pacientes têm maior risco de voltar a ser internados em até 30 dias — isso permite priorizar acompanhamento pós-alta para quem mais precisa. Comparo uma regressão logística (mais simples e interpretável) com uma Random Forest (mais flexível), e escolho com base em recall — para esse problema, prefiro sinalizar risco demais do que de menos, porque um falso negativo aqui é um paciente de alto risco saindo sem nenhum acompanhamento programado.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.risco_readmissao import (
    separar_treino_teste, treinar_random_forest, treinar_logistic_baseline,
    avaliar_modelo, extrair_importancia_features
)
from src.visualization.graficos import plot_matriz_confusao, plot_importancia_features

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

features = pd.read_parquet('../data/processed/features_readmissao.parquet')
print(f"{len(features):,} internações | taxa de readmissão: {features['readmissao_30d'].mean()*100:.1f}%")
features.head()

## 1. Separando treino e teste

Estratifico pela variável alvo porque a readmissão é a classe minoritária (por volta de 20% dos casos) — sem isso, um split de sorte poderia deixar treino e teste com proporções bem diferentes e distorcer as métricas.

In [ ]:
X_treino, X_teste, y_treino, y_teste = separar_treino_teste(features)
print(f'Treino: {len(X_treino):,} | Teste: {len(X_teste):,}')
print(f'Taxa de readmissão no treino: {y_treino.mean()*100:.1f}% | no teste: {y_teste.mean()*100:.1f}%')

## 2. Baseline — Regressão Logística

Padronizo as variáveis antes de treinar, porque a regressão logística é sensível à escala — sem isso, idade e valor_total (em escalas bem diferentes) distorceriam os coeficientes.

In [ ]:
modelo_lr, scaler_lr = treinar_logistic_baseline(X_treino, y_treino)
metricas_lr = avaliar_modelo(modelo_lr, X_teste, y_teste, scaler=scaler_lr)

print(f"AUC: {metricas_lr['auc']:.3f} | Recall: {metricas_lr['recall']:.3f} | Precisão: {metricas_lr['precisao']:.3f} | F1: {metricas_lr['f1']:.3f}")

## 3. Random Forest

Uso `class_weight='balanced'` para compensar o desbalanceamento das classes sem precisar fazer oversampling manual.

In [ ]:
modelo_rf = treinar_random_forest(X_treino, y_treino)
metricas_rf = avaliar_modelo(modelo_rf, X_teste, y_teste)

print(f"AUC: {metricas_rf['auc']:.3f} | Recall: {metricas_rf['recall']:.3f} | Precisão: {metricas_rf['precisao']:.3f} | F1: {metricas_rf['f1']:.3f}")

## 4. Comparando os dois modelos

In [ ]:
comparacao = pd.DataFrame([
    {'modelo': 'Regressão Logística', **{k: v for k, v in metricas_lr.items() if k != 'matriz_confusao'}},
    {'modelo': 'Random Forest', **{k: v for k, v in metricas_rf.items() if k != 'matriz_confusao'}},
])
comparacao

## 5. Matriz de confusão — Random Forest

In [ ]:
fig, ax = plot_matriz_confusao(np.array(metricas_rf['matriz_confusao']))
plt.tight_layout()
plt.savefig('../reports/figures/matriz_confusao_readmissao.png', dpi=150)
plt.show()

## 6. O que mais pesa na decisão do modelo

Idade lidera a importância, seguida do valor da internação (proxy de gravidade e custo do caso) e do histórico de internações anteriores. Faz sentido clinicamente: pacientes mais velhos, com internações mais caras/graves e que já voltaram ao hospital antes são justamente o perfil que a literatura de readmissão hospitalar mais associa a risco.

In [ ]:
importancia = extrair_importancia_features(modelo_rf, X_treino.columns.tolist())

fig, ax = plot_importancia_features(importancia)
plt.tight_layout()
plt.savefig('../reports/figures/importancia_features_readmissao.png', dpi=150)
plt.show()

importancia.head(10)

## Conclusões

O AUC ficou em torno de 0,60 para os dois modelos — moderado, mas coerente com a literatura quando o modelo só tem dados administrativos (idade, especialidade, diagnóstico, custo, permanência) e nenhuma informação clínica mais fina, como comorbidades detalhadas, exames ou medicação em uso. Ainda assim, um recall acima de 50% já ajuda a priorizar quem recebe uma ligação de acompanhamento pós-alta, mesmo sem ser um modelo definitivo.

Para melhorar isso de verdade, o próximo passo seria incluir um índice de comorbidades (Charlson ou Elixhauser) e um histórico de utilização de serviços de saúde mais detalhado do que só a contagem de internações anteriores — informação que os dados administrativos puros do SIH não trazem sozinhos.